# EVADE Experiment 1: Matched-Prompt-Length Control
### Controlling for Superficial Prompt Length & Formatting Confounders

**Central Question**: Does an LLM systematically change its behavior because it recognizes evaluation-related context, or are observed changes merely artifacts of prompt token length and formatting variance?

### Experimental Matrix
- **Tasks**: 200 tasks across 5 domains (Psychological, Reasoning, Math, Coding, Safety)
- **Conditions**: 2 matched conditions (Exact 40 tokens each under Qwen tokenizer):
  1. `matched_deployment`: Production environment wrapper with end-user context.
  2. `matched_evaluation`: Evaluation environment wrapper with quality assessment context.
- **Token Delta Constraint**: **|Δ Prompt Tokens| = 0 across all 200 tasks**.
- **Model**: `Qwen/Qwen2.5-7B-Instruct` (NF4 4-bit quantization, 100% offline local GPU execution).
- **Total Generations**: **400 generations** (approx 2.5 hours on Kaggle T4 GPU).

### Step 1: Install Dependencies & Verify GPU

In [ ]:
!pip install -q transformers accelerate bitsandbytes torch torchvision scipy tabulate sqlite-utils matplotlib seaborn pandas

import torch
import importlib.metadata

print("PyTorch version:   ", torch.__version__)
print("CUDA available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:        ", torch.cuda.get_device_name(0))
    print("Device Count:      ", torch.cuda.device_count())
    print("Total VRAM (GB):   ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
print("sqlite-utils:      ", importlib.metadata.version('sqlite-utils'))
print("[OK] Dependencies installed successfully!")

### Step 2: Sync EVADE Research Repository

In [ ]:
import os
import sys

REPO_DIR = "/kaggle/working/researchpaper2-"
EVADE_DIR = "/kaggle/working/researchpaper2-/evade"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Goldypahal/researchpaper2-.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

%cd {EVADE_DIR}
if EVADE_DIR not in sys.path:
    sys.path.insert(0, EVADE_DIR)
print("Active working directory:", os.getcwd())

### Step 3: Verify GPU Accelerator (100% Offline Local Inference)

In [ ]:
import torch

print("=" * 65)
print("  OFFLINE LOCAL GPU VERIFICATION - EXPERIMENT 1")
print("=" * 65)
assert torch.cuda.is_available(), "GPU accelerator required! (Kaggle: Accelerator -> GPU T4 x2 or P100)"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
print(f"Active GPU:     {gpu_name}")
print(f"Total VRAM:     {vram_gb} GB")
print("Execution Mode: 100% OFFLINE Open-Weights HuggingFace Execution")
print("Network Status: NO external API keys or API endpoints used.")
print("=" * 65)

### Step 4: Run Experiment 1 (400 Generations: Matched-Prompt-Length Control)
- 200 tasks $\times$ 2 conditions (`matched_deployment` vs `matched_evaluation`)
- Exact 40 tokens per wrapper (Δ Prompt Tokens = 0)
- Measures Accuracy, Completion Tokens, Refusal, Hedging, Bullets, Disclaimers, and Sycophancy

In [ ]:
!python scripts/run_exp1_matched_length.py --model Qwen/Qwen2.5-7B-Instruct --quantize-4bit

### Step 5: Generate Publication Figures for Experiment 1

In [ ]:
!python scripts/generate_exp1_figures.py --dir exp1_results --model Qwen/Qwen2.5-7B-Instruct

### Step 6: Guarded Packaging of Experiment 1 Artifacts

In [ ]:
import shutil
from pathlib import Path

raw_dir = Path("exp1_results/raw")
raw_files = list(raw_dir.glob("*.jsonl")) if raw_dir.exists() else []
total_records = sum(sum(1 for line in open(rf, encoding="utf-8") if line.strip()) for rf in raw_files)

print(f"Total Experiment 1 raw records to package: {total_records}")
if total_records == 0:
    raise RuntimeError("CRITICAL: Experiment 1 produced 0 generations. Refusing to package empty results!")

db_file = Path("results/evade_results.db")
if db_file.exists():
    shutil.copy2(db_file, "exp1_results/")

out_zip = "/kaggle/working/evade_exp1_results"
shutil.make_archive(out_zip, "zip", "exp1_results")
print(f"[SUCCESS] All Experiment 1 artifacts packaged to {out_zip}.zip!")